In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import pickle

```
Загрузка датасета
```

In [3]:
df = pd.read_csv('data/drom_archive_cleaned_2018-2025.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 18 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   Название машины   1000000 non-null  str    
 1   Год               1000000 non-null  float64
 2   Цена              1000000 non-null  float64
 3   Объем двигателя   1000000 non-null  float64
 4   Тип двигателя     1000000 non-null  str    
 5   Мощность          1000000 non-null  float64
 6   Коробка передач   1000000 non-null  str    
 7   Привод            1000000 non-null  str    
 8   Пробег            1000000 non-null  float64
 9   Руль              1000000 non-null  str    
 10  Поколение         1000000 non-null  float64
 11  Рестайлинг        1000000 non-null  float64
 12  Тип кузова        1000000 non-null  str    
 13  Метка             1000000 non-null  str    
 14  Город             1000000 non-null  str    
 15  Год объявления    1000000 non-null  int64  
 16  Месяц объявл

## Линейная регрессия

```
Преобразование категориальных признаков
```

In [5]:
categorical = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Поколение', 'Рестайлинг', 'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Месяц объявления', 'Возраст авто']

In [6]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', 'passthrough', numerical)], remainder='drop')

In [7]:
model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])

```
Разделение данных на целевую переменную и признаки
```

In [8]:
y = df['Цена']
X = df.drop('Цена', axis=1)

```
Разделение на обучающую и тестовую выборки (2018-2023, 2024-2025)
```

In [9]:
X_train = X[X['Год объявления'] <= 2023]
X_test = X[X['Год объявления'] >= 2024]

y_train = y[X['Год объявления'] <= 2023]
y_test = y[X['Год объявления'] >= 2024]
joblib.dump((X_train, X_test, y_train, y_test), "data/data_split.pkl")
X_train, X_test, y_train, y_test = joblib.load("data/data_split.pkl")

```
Обучение и сохранение модели
```

In [10]:
model.fit(X_train, y_train)
joblib.dump(model, "models/lr_model.pkl")

['models/lr_model.pkl']

```
Предсказание на тестовой выборке
```

In [11]:
y_pred = model.predict(X_test)

```
Оценка модели
```

In [12]:
lr_mse = mean_squared_error(y_test, y_pred)
lr_rmse = np.sqrt(lr_mse)
lr_mae = mean_absolute_error(y_test, y_pred)
lr_r2 = r2_score(y_test, y_pred)

```
Вывод результатов
```

In [13]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [lr_mse, lr_rmse, lr_mae, lr_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),397_929_774_235.88
1,Среднеквадратическая ошибка (RMSE),630_816.75
2,Средняя абсолютная ошибка (MAE),328_809.37
3,Коэффицент детерминации (R^2),0.62


```
Сохранение метрик в отдельный файл
```

In [14]:
metrics = {
    "model_name": "Linear Regression",
    "MSE": lr_mse,
    "RMSE": lr_rmse,
    "MAE": lr_mae,
    "R2": lr_r2
}

In [15]:
with open("metrics/lr_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)